In [1]:
from pathlib import Path
import sqlite3
from sentence_transformers import CrossEncoder

c:\Users\ronal\source\repos\ARTEFACT-DesafioTecnico\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from scripts.parser_pdf import parserPdf
from scripts.sqlite import criarBanco, consultarSQL
from scripts.embeddings import gerarEmbeddings, consultarEmbeddings
from scripts.bm25 import searchBM25, carregarBM25
from scripts.fusion import Rrf
from scripts.rerank import rerank
from scripts.text_to_sql import textToSQL
from scripts.router import router

In [3]:
query = "Qual é o instrumento mais barato e o mais caro?"

In [9]:
pdf_dir = Path("../data/data_pdf")
csv_dir = Path("../data/data_csv")
chunks_dir = Path("../processed_data/chunks")
bd_dir = "../data/dados.db"
embeddings_dir = "../processed_data/embeddings.npy"

modelo = "BAAI/bge-m3"
modelo_llm = "qwen2.5-coder:7b"
modelo_rerank = "BAAI/bge-reranker-v2-m3"

temperature = 0.3

conexao = sqlite3.connect(bd_dir)

In [5]:
parserPdf(pdf_dir, chunks_dir)
criarBanco(csv_dir, conexao)
gerarEmbeddings(modelo, chunks_dir, embeddings_dir)

Encontrados 1 PDF(s).
Processando: politicas_da_loja.pdf


INFO:sentence_transformers.base.model:No device provided, using cpu


  → 8 chunks criados.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from BAAI/bge-m3.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_t

Carregando: politicas_da_loja.jsonl


Batches: 100%|██████████| 1/1 [00:12<00:00, 12.22s/it]


In [6]:
route = router(query, modelo_llm, temperature)
print(route)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


SQL


In [7]:
def RAG():
    bm25, chunks = carregarBM25(chunks_dir)
    bm25_results = searchBM25(bm25, chunks, query, top_k=5)
    embedding_results = consultarEmbeddings(modelo, embeddings_dir, chunks, query, k=5)

    rrf_results = Rrf(
        [
            embedding_results,
            bm25_results,
        ]
    )

    reranker = CrossEncoder(modelo_rerank)

    final_results = rerank(
        reranker,
        query,
        rrf_results[:10],
        top_k=5,
    )

    return final_results


In [10]:
match route:
    case "RAG":
        print("Executando RAG...")
        resultadoRAG = RAG()

    case "SQL":
        print("Executando Text-to-SQL...")
        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)

    case _:
        resultadoRAG = RAG()
        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)

Executando Text-to-SQL...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [15]:
match route:
    case "RAG":
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

    case "SQL":
        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)

    case _:
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)





--- Resultados do SQL ---
SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Barato' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl ASC
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Caro' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl DESC
    LIMIT 1
);

Colunas: ['instrumento', 'descricao', 'preco', 'tipo']
('Giannini GNF-3 CEQ Elétrico Nylon Natural', 'Violão eletroacústico nylon Giannini com equalizador de 4 bandas. Construção robusta e som amplificado de qualidade profissional.', '1049', 'Mais Barato')
('Tagima TW-7 7 Cordas Aço Natural', 'Violão 7 cordas com cordas de aço para estilos que exigem projeção extra. Braço confortável com tensor regulável.', '999', 'Mais Caro')
